# 2026-10-21_5 MG 堆叠小提琴图——stage 版（ABCB11/TTC39B/TFRC）+ age 版（ABCB11/TTC39B/TFRC/DIO2/APOE）

> **2026-10-25 更新**：应用户要求，**age 版（`_byAge`）先后追加 APOE 与 DIO2**，现为 5 基因
> （ABCB11 → TTC39B → TFRC → DIO2 → APOE，自上而下；顺序沿用用户基因总列表）；
> **stage 版保持 3 基因不变**。两版仍均剔除 70years 与 30years。本 notebook 整体重跑后覆盖同名 PDF。

**需求与口径**

- 细胞范围：仅 MG（Müller glia）
- **剔除供体**：**70years**（仅 macula/central，无 peripheral 对照）与 **30years**（仅 peripheral，无 central 对照）——stage 版与 age 版均剔除
- 两个版本：
  1. **stage 版**（3 基因 ABCB11/TTC39B/TFRC）：X 轴 Early-central; Early-peripheral; Mid-central; Mid-peripheral; Late-central; Late-peripheral（成年页 young-central; young-peripheral; old-central; old-peripheral；young = 21/25/32years，old = 67/75/86years）
  2. **age 版**（5 基因，含 DIO2 与 APOE）：X 轴按 donor_age 逐龄 × central/peripheral 交替——胚胎 12 龄（70→165days）；成年 6 龄（21/25/32/67/75/86years）
- 两版均为：胚胎/成年分页（同一 PDF 页 1 胚胎、页 2 成年）；配色 stage 本色 = central、浅色（lighten 0.45）= peripheral（age 版各龄继承其 stage 颜色，无图例、副标题说明）；sharedY（同一基因两页 y 轴一致，上限跨两期全局计算）
- 仅输出 PDF，不输出 PNG

**数据源（复用每细胞表，不重读大对象）**

- `Data/2026-10-21_3_MG_gene_percell_expr_with70y.rds`（全部 MG 细胞含 70years，7 基因 counts+lognorm；载入后剔除 70/30years）

**输出**

- `Plot/2026-10-21_5_MG_ABCB11_TTC39B_TFRC_stacked_violin_sharedY.pdf`（stage 版，2 页，11×4.5 in，3 基因）
- `Plot/2026-10-21_5_MG_ABCB11_TTC39B_TFRC_stacked_violin_sharedY_byAge.pdf`（age 版，2 页，14×6.5 in，5 基因含 DIO2 与 APOE）

In [ ]:
# ---- 1. 环境与参数 ----------------------------------------------------------
suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(ggplot2)
  library(scales)
})
set.seed(1)

DATE_STR <- "2026-10-21_5"
PROJ_DIR <- "/project/work/2026-10-21_violinplot"
DATA_DIR <- file.path(PROJ_DIR, "Data")
PLOT_DIR <- file.path(PROJ_DIR, "Plot")

PERCELL_RDS <- file.path(DATA_DIR, "2026-10-21_3_MG_gene_percell_expr_with70y.rds")
stopifnot(file.exists(PERCELL_RDS))

# stage 版：3 基因（不变）
GENES_STAGE <- c("ABCB11", "TTC39B", "TFRC")
# age 版：5 基因（2026-10-25 先后追加 APOE、DIO2；顺序沿用用户基因总列表）
GENES_AGE <- c("ABCB11", "TTC39B", "TFRC", "DIO2", "APOE")

# 剔除供体（两版一致）：70years 仅 central、30years 仅 peripheral，均无配对对照
EXCLUDE_AGES <- c("70years", "30years")

# stage 版分组顺序
GROUP_LEVELS_EMB <- c("Early-central", "Early-peripheral", "Mid-central",
                      "Mid-peripheral", "Late-central", "Late-peripheral")
GROUP_LEVELS_ADU <- c("young-central", "young-peripheral", "old-central", "old-peripheral")

# age 版：age 顺序（胚胎 12 龄；成年 6 龄，已剔除 30/70years）
AGE_ORDER_EMB <- c('70days','79days','87days','91days','100days','103days',
                   '116days','137days','141days','142days','162days','165days')
AGE_ORDER_ADU <- c('21years','25years','32years','67years','75years','86years')

# ---- 配色（沿用 panelFH 的 stage/region 逻辑；用户提供）-----------------------
cols.stage.full <- c('#59A14F','#EDC948','#B07AA1','#4E79A7','#E15759')  ## 用户指定统一配色
names(cols.stage.full) <- c('Early','Mid','Late','Young','Old')
cols.stage      <- cols.stage.full[c("Early","Mid","Late")]                       ## dev: Early/Mid/Late
cols.adultstage <- setNames(cols.stage.full[c("Young","Old")], c("young","old"))  ## adult: 数据里为小写 young/old
lighten_col <- function(col, f = 0.45) {
  m <- grDevices::col2rgb(col)
  grDevices::rgb(t(m + (255 - m) * f), maxColorValue = 255)
}
## central = 本色, peripheral = 浅色
region_cols <- function(base) {
  v <- c(base, sapply(base, lighten_col))
  names(v) <- c(paste0(names(base), "_central"), paste0(names(base), "_peripheral"))
  v
}

cols.emb.groups <- region_cols(cols.stage)
names(cols.emb.groups) <- gsub("_", "-", names(cols.emb.groups))
cols.adu.groups <- region_cols(cols.adultstage)
names(cols.adu.groups) <- gsub("_", "-", names(cols.adu.groups))

cat("stage 版基因（3）:", paste(GENES_STAGE, collapse = " -> "), "\n")
cat("age 版基因（5，含追加的 DIO2 与 APOE）:", paste(GENES_AGE, collapse = " -> "), "\n")
cat("剔除供体:", paste(EXCLUDE_AGES, collapse = ", "), "（stage 版与 age 版一致）\n")
cat("\nstage 版分组配色：\n")
print(cols.emb.groups[GROUP_LEVELS_EMB])
print(cols.adu.groups[GROUP_LEVELS_ADU])
cat("\nage 版成年龄序（", length(AGE_ORDER_ADU), "龄）:", paste(AGE_ORDER_ADU, collapse = ", "), "\n")

In [ ]:
# ---- 2. 读取每细胞表，剔除 70/30years，计算各自版本 sharedY 上限 ---------------
percell_all <- readRDS(PERCELL_RDS)
stopifnot(all(paste0(GENES_AGE, "_lognorm") %in% colnames(percell_all)))
cat("载入 MG 总细胞数（含 70years）:", nrow(percell_all), "\n")

## 剔除 70years / 30years（记录剔除数）
excl_tab <- table(percell_all$donor_age[percell_all$donor_age %in% EXCLUDE_AGES])
cat("剔除细胞数：\n"); print(excl_tab)
percell <- percell_all[!(percell_all$donor_age %in% EXCLUDE_AGES), ]
stopifnot(!any(percell$donor_age %in% EXCLUDE_AGES))
percell$period <- factor(as.character(percell$period), levels = c("Embryonic", "Adult"))
emb <- percell[percell$period == "Embryonic", ]
adu <- percell[percell$period == "Adult", ]
cat("\n剔除后 MG 细胞数：胚胎", nrow(emb), "| 成年", nrow(adu),
    "| 合计", nrow(percell), "\n")

cat("\n[Embryonic] stage 版分组细胞数：\n")
print(table(droplevels(factor(as.character(emb$stage_region), levels = GROUP_LEVELS_EMB))))
cat("\n[Adult] stage 版分组细胞数：\n")
print(table(droplevels(factor(as.character(adu$stage_region), levels = GROUP_LEVELS_ADU))))

## age 版分组变量：age_region = "age\nregion"
percell$age_region <- paste0(percell$donor_age, "\n", percell$region)
emb <- percell[percell$period == "Embryonic", ]
adu <- percell[percell$period == "Adult", ]
make_age_levels <- function(ages) {
  unlist(lapply(ages, function(a) c(paste0(a, "\ncentral"), paste0(a, "\nperipheral"))))
}
GROUP_AGE_EMB <- make_age_levels(AGE_ORDER_EMB)
GROUP_AGE_ADU <- make_age_levels(AGE_ORDER_ADU)
cat("\n[Embryonic] age 版分组细胞数（12 龄 × 2）：\n")
print(table(factor(emb$age_region, levels = GROUP_AGE_EMB)))
cat("\n[Adult] age 版分组细胞数（6 龄 × 2）：\n")
print(table(factor(adu$age_region, levels = GROUP_AGE_ADU)))

## sharedY 上限：剔除后跨两期全局最大值；stage 版与 age 版各用其基因集计算
gene_y_max_stage <- vapply(GENES_STAGE, function(g) max(percell[[paste0(g, "_lognorm")]]), numeric(1))
gene_y_max_age   <- vapply(GENES_AGE,   function(g) max(percell[[paste0(g, "_lognorm")]]), numeric(1))
cat("\nstage 版每基因跨两期全局最大 lognorm：\n"); print(round(gene_y_max_stage, 3))
cat("\nage 版每基因跨两期全局最大 lognorm（含 DIO2 与 APOE）：\n"); print(round(gene_y_max_age, 3))

build_mini_seurat <- function(df, group_var, group_levels, genes) {
  counts_mat <- Matrix::t(as(as.matrix(df[, paste0(genes, "_counts")]), "dgCMatrix"))
  data_mat   <- Matrix::t(as(as.matrix(df[, paste0(genes, "_lognorm")]), "dgCMatrix"))
  rownames(counts_mat) <- genes; colnames(counts_mat) <- df$cell
  rownames(data_mat)   <- genes; colnames(data_mat)   <- df$cell
  mini_meta <- data.frame(grp = factor(as.character(df[[group_var]]), levels = group_levels),
                          row.names = df$cell)
  colnames(mini_meta) <- group_var
  mini <- CreateSeuratObject(counts = counts_mat, meta.data = mini_meta)
  mini[["RNA"]]$data <- data_mat
  Idents(mini) <- group_var
  mini
}

In [ ]:
# ---- 3. stage 版（3 基因）：构建两期堆叠小提琴（同一基因两页 y 轴一致）---------
make_stacked_violin_sharedY <- function(df, group_levels, group_cols, title_str) {
  mini <- build_mini_seurat(df, "stage_region", group_levels, GENES_STAGE)

  blank_df <- data.frame(
    feature    = factor(GENES_STAGE, levels = GENES_STAGE),
    ident      = group_levels[1],
    expression = gene_y_max_stage + pmax(0.001 * gene_y_max_stage, 0.01)
  )

  p <- VlnPlot(mini, features = GENES_STAGE, group.by = "stage_region",
               stack = TRUE, fill.by = "ident",
               cols = group_cols[group_levels],
               pt.size = 0, flip = TRUE) +
    geom_blank(data = blank_df, aes(x = ident, y = expression), inherit.aes = FALSE) +
    scale_y_continuous(expand = c(0, 0), limits = c(0, NA), oob = squish,
                       labels = function(x) c(rep("", length(x) - 2), x[length(x) - 1], "")) +
    labs(title = title_str,
         subtitle = sprintf("MG cells: n = %s | young = 21/25/32y, old = 67/75/86y | per-gene y-axis shared between periods",
                            format(nrow(df), big.mark = ","))) +
    theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 13),
          plot.subtitle = element_text(hjust = 0.5, size = 8.5))

  ## 核对：X 轴分组顺序 / 面板基因顺序 / 填充色 / 每面板 y 范围
  bd <- ggplot_build(p)
  bp <- bd$layout$panel_params[[1]]
  cat(sprintf("[%s] X 轴（分组，左->右）: %s\n", title_str,
              paste(bp$x$get_labels(), collapse = " | ")))
  cat(sprintf("[%s] 面板基因顺序（自上而下）: %s\n", title_str,
              paste(levels(factor(p$data$feature, levels = GENES_STAGE)), collapse = " | ")))
  fills <- unique(bd$data[[1]][order(bd$data[[1]]$x), "fill"])
  stopifnot(identical(fills, unname(group_cols[group_levels])))
  yr <- do.call(rbind, lapply(bd$layout$panel_params, function(pp) pp$y$get_limits()))
  rownames(yr) <- GENES_STAGE
  cat(sprintf("[%s] 每基因 y 轴范围：\n", title_str)); print(round(yr, 4))
  invisible(list(plot = p, yranges = yr))
}

vln_emb <- make_stacked_violin_sharedY(emb, GROUP_LEVELS_EMB, cols.emb.groups, "Embryonic MG")
vln_adu <- make_stacked_violin_sharedY(adu, GROUP_LEVELS_ADU, cols.adu.groups, "Adult MG")

## 校验：两页每基因 y 轴范围逐一相等
cat("\n[stage 版] 两页 y 轴范围是否一致（逐基因）：\n")
ok <- vapply(GENES_STAGE, function(g) isTRUE(all.equal(vln_emb$yranges[g, ], vln_adu$yranges[g, ])),
             logical(1))
print(ok); stopifnot(all(ok))
cat("[stage 版] 校验通过：同一基因胚胎页与成年页 y 轴范围完全一致\n")

print(vln_emb$plot)
print(vln_adu$plot)

In [ ]:
# ---- 4. 输出 stage 版 PDF（页 1 胚胎 / 页 2 成年）-----------------------------
pdf_stage <- file.path(PLOT_DIR, paste0(DATE_STR, "_MG_ABCB11_TTC39B_TFRC_stacked_violin_sharedY.pdf"))
pdf(pdf_stage, width = 11, height = 4.5, useDingbats = FALSE)
print(vln_emb$plot)
print(vln_adu$plot)
dev.off()
cat("已输出 stage 版 PDF（2 页）:", basename(pdf_stage), "\n")
cat("文件大小 (MB):", round(file.size(pdf_stage) / 1024^2, 3), "\n")

In [ ]:
# ---- 5. age 版（5 基因含 DIO2 与 APOE）：各龄继承 stage 颜色，构建两期堆叠小提琴 --
## 颜色：各 age 继承其 stage 颜色（central = 本色，peripheral = 浅色 45%）
cols_all_stage <- c(cols.stage, cols.adultstage)
make_age_cols <- function(ages, df) {
  st <- vapply(ages, function(a) unique(as.character(df$stage[df$donor_age == a]))[1], "")
  stopifnot(all(st %in% names(cols_all_stage)))
  base <- cols_all_stage[st]
  v <- unlist(lapply(base, function(b) c(b, lighten_col(b))))
  setNames(v, make_age_levels(ages))
}
cols.age.emb <- make_age_cols(AGE_ORDER_EMB, percell)
cols.age.adu <- make_age_cols(AGE_ORDER_ADU, percell)

cat("胚胎 12 龄对应 stage：\n")
print(vapply(AGE_ORDER_EMB, function(a) unique(as.character(percell$stage[percell$donor_age == a]))[1], ""))
cat("\n成年 6 龄对应 stage：\n")
print(vapply(AGE_ORDER_ADU, function(a) unique(as.character(percell$stage[percell$donor_age == a]))[1], ""))

make_age_violin_page <- function(df, group_levels, group_cols, title_str) {
  mini <- build_mini_seurat(df, "age_region", group_levels, GENES_AGE)

  present <- group_levels[group_levels %in% unique(as.character(df$age_region))]
  if (length(present) < length(group_levels)) {
    cat(sprintf("[%s] 空分组（无细胞，保留空槽位）: %s\n", title_str,
                paste(group_levels[!group_levels %in% present], collapse = " | ")))
  }
  blank_df <- data.frame(feature = factor(GENES_AGE, levels = GENES_AGE), ident = present[1],
                         expression = gene_y_max_age + pmax(0.001 * gene_y_max_age, 0.01))

  p <- VlnPlot(mini, features = GENES_AGE, group.by = "age_region",
               stack = TRUE, fill.by = "ident",
               cols = group_cols[group_levels], pt.size = 0, flip = TRUE) +
    geom_blank(data = blank_df, aes(x = ident, y = expression), inherit.aes = FALSE) +
    scale_x_discrete(drop = FALSE) +
    scale_y_continuous(expand = c(0, 0), limits = c(0, NA), oob = squish,
                       labels = function(x) c(rep("", length(x) - 2), x[length(x) - 1], "")) +
    labs(title = title_str,
         subtitle = sprintf("MG cells: n = %s | fill = stage color (dark central / light peripheral) | per-gene y-axis shared between periods",
                            format(nrow(df), big.mark = ","))) +
    theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 13),
          plot.subtitle = element_text(hjust = 0.5, size = 8.5),
          axis.text.x = element_text(angle = 0, size = 6.8),
          legend.position = "none")

  ## 核对：X 轴槽位数 / 面板基因顺序 / 填充色逐槽位 / 每面板 y 范围
  bd <- ggplot_build(p)
  bp <- bd$layout$panel_params[[1]]
  x_levels <- bp$x$get_labels()
  cat(sprintf("[%s] X 轴分组数: %d（首个: %s | 末个: %s）\n", title_str,
              length(x_levels), gsub("\n", "-", x_levels[1]), gsub("\n", "-", tail(x_levels, 1))))
  cat(sprintf("[%s] 面板基因顺序（自上而下）: %s\n", title_str,
              paste(levels(factor(p$data$feature, levels = GENES_AGE)), collapse = " | ")))
  ## byAge 版各 age 继承 stage 颜色 -> 多分组同色，逐 x 槽位核对填充色（空槽位 NA 跳过）
  fill_by_x <- vapply(seq_along(x_levels), function(i) {
    v <- as.character(bd$data[[1]]$fill[bd$data[[1]]$x == i])
    if (length(v)) unique(v)[1] else NA_character_
  }, character(1))
  expected <- unname(as.character(group_cols[x_levels]))
  hit <- !is.na(fill_by_x)
  stopifnot(all(fill_by_x[hit] == expected[hit]), sum(hit) == length(present))
  cat(sprintf("[%s] 填充色逐槽位校验通过（%d/%d 槽位有细胞）\n", title_str,
              sum(hit), length(x_levels)))
  yr <- do.call(rbind, lapply(bd$layout$panel_params, function(pp) pp$y$get_limits()))
  rownames(yr) <- GENES_AGE
  cat(sprintf("[%s] 每基因 y 轴范围：\n", title_str)); print(round(yr, 4))
  invisible(list(plot = p, yranges = yr))
}

vln_age_emb <- make_age_violin_page(emb, GROUP_AGE_EMB, cols.age.emb, "Embryonic MG (by age)")
vln_age_adu <- make_age_violin_page(adu, GROUP_AGE_ADU, cols.age.adu, "Adult MG (by age)")

## 校验：两页每基因 y 轴范围逐一相等
cat("\n[age 版] 两页 y 轴范围是否一致（逐基因，含 DIO2 与 APOE）：\n")
ok <- vapply(GENES_AGE, function(g) isTRUE(all.equal(vln_age_emb$yranges[g, ], vln_age_adu$yranges[g, ])),
             logical(1))
print(ok); stopifnot(all(ok))
cat("[age 版] 校验通过：同一基因胚胎页与成年页 y 轴范围完全一致\n")
## 交叉校验：两版共有的 3 个基因上限基准一致（同一套细胞）
ok2 <- vapply(GENES_STAGE, function(g) isTRUE(all.equal(vln_emb$yranges[g, ], vln_age_emb$yranges[g, ])),
              logical(1))
cat("[交叉校验] stage 版与 age 版共有基因 y 轴范围一致：", all(ok2), "\n"); stopifnot(all(ok2))

print(vln_age_emb$plot)
print(vln_age_adu$plot)

In [ ]:
# ---- 6. 输出 age 版 PDF（页 1 胚胎 / 页 2 成年；5 基因含 DIO2 与 APOE）---------
pdf_age <- file.path(PLOT_DIR, paste0(DATE_STR, "_MG_ABCB11_TTC39B_TFRC_stacked_violin_sharedY_byAge.pdf"))
pdf(pdf_age, width = 14, height = 6.5, useDingbats = FALSE)
print(vln_age_emb$plot)
print(vln_age_adu$plot)
dev.off()
cat("已输出 age 版 PDF（2 页，5 基因含 DIO2 与 APOE）:", basename(pdf_age), "\n")
cat("文件大小 (MB):", round(file.size(pdf_age) / 1024^2, 3), "\n")

In [ ]:
# ---- 7. 文件清单与 sessionInfo -----------------------------------------------
fi <- data.frame(
  file = c(basename(pdf_stage), basename(pdf_age)),
  size_MB = round(c(file.size(pdf_stage), file.size(pdf_age)) / 1024^2, 3))
print(fi, row.names = FALSE)
cat("\n全部完成。\n\n")
sessionInfo()